In [1]:
import os

In [2]:
%pwd

'/home/user/Documents/Text_Summarization/research'

In [3]:
os.chdir("..")

In [4]:
%pwd

'/home/user/Documents/Text_Summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [8]:
import os
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

/home/user/Documents/Text_Summarization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    
    def convert_examples_to_features(self, example_batch):
      input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024,    truncation=True)

      target_encodings = self.tokenizer(   text_target=example_batch['summary'],  max_length=128,  truncation=True  )

      return { 'input_ids': input_encodings['input_ids'], 'attention_mask': input_encodings['attention_mask'], 'labels': target_encodings['input_ids'] }
    

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)

        dataset_samsum["train"] = dataset_samsum["train"].select(range(100))
        dataset_samsum["validation"] = dataset_samsum["validation"].select(range(20))
 
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)

        dataset_samsum_pt.save_to_disk(
        os.path.join(self.config.root_dir, "samsum_dataset")  )

In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-04-26 21:22:20,122: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-26 21:22:20,129: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-26 21:22:20,135: INFO: common: created directory at: artifacts]
[2026-04-26 21:22:20,137: INFO: common: created directory at: artifacts/data_transformation]


Saving the dataset (1/1 shards): 100%|██████████| 20/20 [00:00<00:00, 1281.37 examples/s]
